In [34]:
from sympy import *
init_printing()
# import numpy as np



# **Uge 1**

In [ ]:

imax_symbol = symbols("imax")


def block_sum(mat_list, imax_val):

    if len(mat_list) == 1:
        total = mat_list[0] * imax_val
    else:
        total = sum(mat_list, zeros(3, 3))
    return total.subs(imax_symbol, imax_val)


def calc_sums(A_list, B_list, D_list, imax_val):
    A = block_sum(A_list, imax_val)
    B = block_sum(B_list, imax_val)
    D = block_sum(D_list, imax_val)

    return Matrix(BlockMatrix([[A, B],
                               [B, D]]))


def solve_sys(matrix, n, m):
    
    rhs = Matrix.vstack(n, m)

    return matrix.inv() * rhs


A_i = Matrix([
    [1/imax_symbol,      0,      0],
    [     0, 1/imax_symbol,      0],
    [     0,      0, 1/imax_symbol],
])

B_i = zeros(3, 3)

D_1 = Rational(13, 12) * Rational(127, 100)**3 * Matrix([
    [147.87,   2.81,   0],
    [  2.81,   9.05,   0],
    [     0,      0, 3.3],
])

D_3 = D_1   # D_1 = D_3

D_2 = Rational(1, 12) * Rational(127, 100)**3 * Matrix([
    [  9.05,   2.81,   0],
    [  2.81, 147.87,   0],
    [     0,      0, 3.3],
])

n = Matrix([0, 0, 0])       
m = Matrix([1000, 0, 0])    



imax_val = 3
sol = solve_sys(
    calc_sums([A_i], [B_i],
              [D_1, D_2, D_3], imax_val),
    n, m,
)
sol


# **Uge 2**

In [18]:
import numpy as np

def constitutive_matrix(E1, E2, G12, nu12):
    S_2D = np.array([
        [ 1.0/E1,   -nu12/E1,  0.0    ],
        [-nu12/E1,   1.0/E2,   0.0    ],
        [ 0.0,       0.0,      1.0/G12],
    ])
    return S_2D





def micromechanics(Ef, nuf, Em, num, vf, xi_E2=1.0, xi_G12=1.0):
    """Effective elastic properties of an aligned fibre composite ply.

    Inputs
    ------
    Ef, nuf : fibre Young's modulus and Poisson's ratio
    Em, num : matrix Young's modulus and Poisson's ratio
    vf      : fibre volume fraction (0..1)
    xi_E2   : Halpin-Tsai geometry factor for E2  (default 2)
    xi_G12  : Halpin-Tsai geometry factor for G12 (default 1)

    Returns a dict with the 5 constants and the matrices S_3D, S_2D, Q_2D.
    (Keep all moduli in the same unit, e.g. all GPa.)
    """
    vm = 1.0 - vf                       # matrix volume fraction

    # constituent shear moduli (isotropic)
    Gf = Ef / (2.0 * (1.0 + nuf))
    Gm = Em / (2.0 * (1.0 + num))

    # --- 1) the 5 independent stiffness parameters ------------------------
    # E1: rule of mixtures
    E1 = vf * Ef + vm * Em

    # E2: Halpin-Tsai
    eta_E = (Ef / Em - 1.0) / (Ef / Em + xi_E2)
    E2 = Em * (1.0 + xi_E2 * eta_E * vf) / (1.0 - eta_E * vf)

    # G12: Halpin-Tsai
    eta_G = (Gf / Gm - 1.0) / (Gf / Gm + xi_G12)
    G12 = Gm * (1.0 + xi_G12 * eta_G * vf) / (1.0 - eta_G * vf)

    # nu12: rule of mixtures
    nu12 = vf * nuf + vm * num

    # nu23: via bulk modulus (compressibility approach)
    nu21 = nu12 * E2 / E1               # reciprocal relation
    Kf = Ef / (3.0 * (1.0 - 2.0 * nuf))
    Km = Em / (3.0 * (1.0 - 2.0 * num))
    K = 1.0 / (vf / Kf + vm / Km)       # equal-stress mixture
    nu23 = 1.0 - nu21 - E2 / (3.0 * K)

    # dependent constants (transverse isotropy)
    E3 = E2
    nu13 = nu12
    G13 = G12
    G23 = E2 / (2.0 * (1.0 + nu23))

    # --- 2) compliance matrix S ------------------------------------------
    # full 3D (6x6), order 1, 2, 3, 4=23, 5=13, 6=12
    S_3D = np.array([
        [ 1.0/E1,    -nu12/E1,  -nu13/E1,  0.0,      0.0,      0.0    ],
        [-nu12/E1,    1.0/E2,   -nu23/E2,  0.0,      0.0,      0.0    ],
        [-nu13/E1,   -nu23/E2,   1.0/E3,   0.0,      0.0,      0.0    ],
        [ 0.0,        0.0,       0.0,      1.0/G23,  0.0,      0.0    ],
        [ 0.0,        0.0,       0.0,      0.0,      1.0/G13,  0.0    ],
        [ 0.0,        0.0,       0.0,      0.0,      0.0,      1.0/G12],
    ])

    # plane stress (3x3), order 1, 2, 6=12
    S_2D = np.array([
        [ 1.0/E1,   -nu12/E1,  0.0    ],
        [-nu12/E1,   1.0/E2,   0.0    ],
        [ 0.0,       0.0,      1.0/G12],
    ])

    # --- 3) inverted constitutive matrix Q = inv(S_2D) -------------------
    Q_2D = np.linalg.inv(S_2D)

    return {
        "E1": E1, "E2": E2, "G12": G12, "nu12": nu12, "nu23": nu23,
        "S_3D": S_3D, "S_2D": S_2D, "Q_2D": Q_2D,
    }


# if __name__ == "__main__":
#     # Exercise 1 test case (example 2.1)
#     res = micromechanics(Ef=76.0, nuf=0.20, Em=4.0, num=0.30, vf=0.55)

#     print("5 independent stiffness parameters:")
#     print(f"  E1   = {res['E1']:.4f} GPa")
#     print(f"  E2   = {res['E2']:.4f} GPa")
#     print(f"  G12  = {res['G12']:.4f} GPa")
#     print(f"  nu12 = {res['nu12']:.4f}")
#     print(f"  nu23 = {res['nu23']:.4f}")

#     np.set_printoptions(precision=5, suppress=True)
#     print("\nS_3D (compliance, 6x6) [1/GPa]:")
#     display(res["S_3D"])
#     print("\nS_2D (plane-stress compliance, 3x3) [1/GPa]:")
#     display(res["S_2D"])
#     print("\nQ_2D (plane-stress stiffness = inv(S_2D), 3x3) [GPa]:")
#     display(res["Q_2D"])


res = micromechanics(Ef=76.0, nuf=0.20, Em=4.0, num=0.30, vf=0.55)
res["Q_2D"]

array([[44.32257,  2.94927,  0.     ],
       [ 2.94927, 12.03783,  0.     ],
       [ 0.     ,  0.     ,  4.60353]])

# **Uge 3**

In [35]:
import matplotlib.pyplot as plt
 

def Build_T(theta):
    c, s = np.cos(np.radians(theta)), np.sin(np.radians(theta))
    return np.array([[c*c, s*s, -2*s*c],
                     [s*s, c*c,  2*s*c],
                     [s*c, -s*c, c*c - s*s]])


def Build_Q_global(Q_local, theta):
    T = Build_T(theta)
    return T @ Q_local @ T.T



# Find S matricen
eks_3_1_S_matrix = constitutive_matrix(E1=181, E2=10.3, G12=7.17, nu12=0.28)

# Find den local Q matrice, ved at invertere S
Q_matrix_local = np.linalg.inv(eks_3_1_S_matrix)

# Find den globale Q matrice
Q_global_eks_3_1 = Build_Q_global(Q_matrix_local, 30)
Q_global_eks_3_1



# Function til at plotte G_global, given a local Q matrix and a range of theta values
def plot_Q_global_sweep(Q_local, n_points=361, figsize=(8, 5)):
    thetas = np.linspace(-90, 90, n_points)
    Q_vals = np.array([Build_Q_global(Q_local, t) for t in thetas])

    labels = {
        (0, 0): "Q11", (0, 1): "Q12", (0, 2): "Q16",
        (1, 1): "Q22", (1, 2): "Q26", (2, 2): "Q66",
    }

    fig, ax = plt.subplots(figsize=figsize)
    for (i, j), label in labels.items():
        ax.plot(thetas, Q_vals[:, i, j], label=label)

    ax.set_xlabel("theta [deg]")
    ax.set_ylabel("Q [GPa]")
    ax.set_xlim(-90, 90)
    ax.set_title("Global stiffness matrix components vs. ply angle")
    ax.legend()
    ax.grid(True)
    fig.tight_layout()
    return fig, ax


# plot_Q_global_sweep(Q_matrix_local)
# plt.show()


# Sigmaer skrives som GPa
# Find global strains
def find_epsilons_given_sigma(Q_local, sigma_x, sigma_y, tau_xy, theta):
    Q_global = Build_Q_global(Q_local, theta)
    sigma = np.array([sigma_x, sigma_y, tau_xy])
    epsilon = np.linalg.inv(Q_global) @ sigma

    return epsilon

# Applied stress with only one non zero component, sigma_x = 0.1 GPa
find_epsilons_given_sigma(Q_matrix_local, sigma_x=0.1, sigma_y=0, tau_xy=0, theta=30)

# Applied stress with more than one non-zero stress component
find_epsilons_given_sigma(Q_matrix_local, sigma_x=0.1, sigma_y=0.3, tau_xy=0, theta=30)

# Applied stress specific stresses
epislon_global  = find_epsilons_given_sigma(Q_matrix_local, sigma_x=2, sigma_y=1, tau_xy=0.5, theta=30)


# Plotter epsilon som funktion a theta
def plot_epsilons_vs_theta(Q_local, sigma_x, sigma_y, tau_xy, theta_range=(-90, 90), n_points=361, figsize=(8, 5)):
    thetas = np.linspace(theta_range[0], theta_range[1], n_points)
    epsilons = np.array([
        find_epsilons_given_sigma(Q_local, sigma_x, sigma_y, tau_xy, t)
        for t in thetas
    ])

    labels = ["eps_x", "eps_y", "gamma_xy"]

    fig, ax = plt.subplots(figsize=figsize)
    for i, label in enumerate(labels):
        ax.plot(thetas, epsilons[:, i], label=label)

    ax.set_xlabel("theta [deg]")
    ax.set_ylabel("epsilon")
    ax.set_xlim(theta_range)
    ax.set_title(f"Strains vs. ply angle (sigma_x={sigma_x}, sigma_y={sigma_y}, tau_xy={tau_xy})")
    ax.legend()
    ax.grid(True)
    fig.tight_layout()
    return fig, ax


# plot_epsilons_vs_theta(Q_matrix_local, sigma_x=0.1, sigma_y=0, tau_xy=0)
# plt.show()


def get_local_strain_from_global_strain(epsilon_global, theta):
   
    return Build_T(theta).T @ np.asarray(epsilon_global, dtype=float)


get_local_strain_from_global_strain(epislon_global, theta=30)

array([ 0.0108 ,  0.07594, -0.02552])